# 03 — RAG Chain & Model Serving Deployment

**Purpose:** Wire the Vector Search retriever to a Foundation Model API chat model to answer
questions about the insurance data, then deploy that chain as a real Mosaic AI Model Serving
endpoint — this is the "RAG served with Mosaic AI Model Serving" deliverable.

### What This Notebook Does
1. Defines an `mlflow.pyfunc.ChatAgent` that retrieves top-k policy documents and grounds a
   Foundation Model API chat completion in them.
2. Smoke-tests the agent locally, then logs and registers it to Unity Catalog with MLflow.
3. Deploys the registered model via the Agent Framework to a Mosaic AI Model Serving
   endpoint and queries it live to prove it works end-to-end.

### Source & Target
| Direction | Resource |
|-----------|----------|
| Source | `insurance_rag_agent.docs.policy_documents_index` |
| Target | UC model `insurance_rag_agent.agent_tools.insurance_rag_model` + serving endpoint |

> **Prerequisites:** Run `src/02_vector_search_index` first. Deploying consumes Free
> Edition's limited model-serving quota — stop/delete the endpoint after demoing.

---

In [ ]:
%pip install -U mlflow databricks-agents databricks-vectorsearch "databricks-sdk[openai]" openai
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG               = 'insurance_rag_agent'
VS_ENDPOINT_NAME       = 'insurance_rag_agent_vs_endpoint'
INDEX_NAME             = f'{CATALOG}.docs.policy_documents_index'
# Get CHAT_MODEL_ENDPOINT and MODEL_QUERY_BASE_PATH from Serving > (your chat model) >
# "Use this model" > the model= and base_url= values in the generated code snippet.
# AI-Gateway/UC-governed workspaces show something like:
#   model='insurance_rag_agent.agent_tools.meta-llama-3-3-70b-instruct_backend'
#   base_url='https://<host>/ai-gateway/mlflow/v1'
# Classic Foundation Model API workspaces show something like:
#   model='databricks-meta-llama-3-3-70b-instruct'
#   base_url='https://<host>/serving-endpoints'
CHAT_MODEL_ENDPOINT    = f'{CATALOG}.agent_tools.meta-llama-3-3-70b-instruct_backend'
MODEL_QUERY_BASE_PATH  = '/ai-gateway/mlflow/v1'
RAG_MODEL_NAME         = f'{CATALOG}.agent_tools.insurance_rag_model'
RAG_ENDPOINT_NAME      = 'insurance_rag_endpoint'
NUM_RESULTS            = 3
SAMPLE_QUESTION        = 'What do the records show about smokers in the southeast region?'

print(f'Index: {INDEX_NAME}')
print(f'Chat model endpoint: {CHAT_MODEL_ENDPOINT}')
print(f'Registered model: {RAG_MODEL_NAME}')

In [0]:
import uuid

import mlflow
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse

mlflow.set_registry_uri('databricks-uc')

---

In [ ]:
class InsuranceRAGAgent(ChatAgent):
    '''Retrieves relevant policy documents, then grounds a Foundation Model API chat
    completion in them. Clients are created inside predict() rather than __init__ so
    the agent instance stays picklable for MLflow logging.'''

    def __init__(self, endpoint_name, index_name, chat_model_endpoint, base_path, num_results):
        self.endpoint_name       = endpoint_name
        self.index_name          = index_name
        self.chat_model_endpoint = chat_model_endpoint
        self.base_path           = base_path
        self.num_results         = num_results

    def predict(self, messages, context=None, custom_inputs=None):
        from databricks.sdk import WorkspaceClient
        from databricks.vector_search.client import VectorSearchClient
        from openai import OpenAI

        question = messages[-1].content

        w            = WorkspaceClient()
        bearer_token = w.config.authenticate()['Authorization'].split(' ', 1)[1]

        # VectorSearchClient() with no args does its OWN credential auto-detection via
        # MLflow's tracking URI, which isn't configured inside a served-model container even
        # though `resources=` is declared below — that mismatch is what throws "the MLflow
        # tracking URI was set to 'None'". Passing the workspace URL and token explicitly
        # (the same bearer token used for the chat client below) bypasses that lookup entirely.
        vsc = VectorSearchClient(
            workspace_url         = w.config.host,
            personal_access_token = bearer_token,
            disable_notice        = True,
        )
        index = vsc.get_index(endpoint_name=self.endpoint_name, index_name=self.index_name)
        results = index.similarity_search(
            query_text  = question,
            columns     = ['policy_text'],
            num_results = self.num_results,
        )
        context_text = '\n'.join(row[0] for row in results['result']['data_array'])

        # CHAT_MODEL_ENDPOINT is queried through this workspace's AI Gateway path, not the
        # classic /serving-endpoints path — w.serving_endpoints.get_open_ai_client() can't be
        # redirected there, so this stays a manually-built OpenAI client. It's still backed by
        # a properly *scoped* credential though: log_model() below declares this endpoint (and
        # the vector index) as `resources`, so Databricks injects a managed credential for them
        # into the deployed container, and config.authenticate() resolves it — no ambient
        # notebook-only auth fallback once this model is actually deployed.
        openai_client = OpenAI(
            api_key  = bearer_token,
            base_url = f'{w.config.host}{self.base_path}',
        )
        completion = openai_client.chat.completions.create(
            model    = self.chat_model_endpoint,
            messages = [
                {'role': 'system', 'content': f'Answer using only this context:\n{context_text}'},
                {'role': 'user',   'content': question},
            ],
        )
        answer = completion.choices[0].message.content

        return ChatAgentResponse(messages=[
            ChatAgentMessage(id=str(uuid.uuid4()), role='assistant', content=answer)
        ])


rag_agent = InsuranceRAGAgent(
    endpoint_name       = VS_ENDPOINT_NAME,
    index_name          = INDEX_NAME,
    chat_model_endpoint = CHAT_MODEL_ENDPOINT,
    base_path           = MODEL_QUERY_BASE_PATH,
    num_results         = NUM_RESULTS,
)

smoke_test = rag_agent.predict(
    [ChatAgentMessage(id=str(uuid.uuid4()), role='user', content=SAMPLE_QUESTION)]
)
print(smoke_test.messages[-1].content)

---

In [ ]:
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksVectorSearchIndex

# Declaring the agent's dependencies lets Databricks inject scoped, auto-rotated credentials
# for them into the deployed container, instead of the agent falling back to ambient auth.
resources = [
    DatabricksVectorSearchIndex(index_name=INDEX_NAME),
    DatabricksServingEndpoint(endpoint_name=CHAT_MODEL_ENDPOINT),
]

with mlflow.start_run(run_name='insurance_rag_agent'):
    logged_model = mlflow.pyfunc.log_model(
        name                  = 'rag_chain',
        python_model          = rag_agent,
        registered_model_name = RAG_MODEL_NAME,
        resources             = resources,
    )

model_version = logged_model.registered_model_version
print(f'Registered {RAG_MODEL_NAME} version {model_version}')

---

In [ ]:
from databricks import agents
from mlflow import MlflowClient

# Deploy whatever is actually newest in Unity Catalog, not whatever `model_version` happens
# to hold in this session — if the logging cell above wasn't the last thing run before this
# one (e.g. after a Python restart), model_version could be stale or undefined, and this
# guarantees the deploy targets the real latest version regardless.
latest_version = max(
    int(mv.version) for mv in MlflowClient().search_model_versions(f"name='{RAG_MODEL_NAME}'")
)
print(f'Latest registered version: {latest_version}')

deployment_info = agents.deploy(
    model_name    = RAG_MODEL_NAME,
    model_version = latest_version,
    endpoint_name = RAG_ENDPOINT_NAME,
    scale_to_zero = True,  # required on Free Edition — no provisioned/always-on throughput
)

print(f'Deployed endpoint: {deployment_info.endpoint_name}')

---

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# get_open_ai_client() targets the classic /serving-endpoints path with zero manual token
# plumbing — a clean fit here since insurance_rag_endpoint is our own real Model Serving
# deployment (unlike CHAT_MODEL_ENDPOINT above, which lives behind this workspace's AI
# Gateway path and needs the manual client instead). Uses RAG_ENDPOINT_NAME (a static config
# constant) rather than deployment_info.endpoint_name, so this cell works even in a fresh
# session where the deploy cell above hasn't been re-run — the endpoint already exists in
# Databricks once deployed once.
serving_client = w.serving_endpoints.get_open_ai_client()

live_response = serving_client.chat.completions.create(
    model    = RAG_ENDPOINT_NAME,
    messages = [{'role': 'user', 'content': SAMPLE_QUESTION}],
)

print(live_response.choices[0].message.content)